[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/3-2_RUL%EC%98%88%EC%B8%A11_%EC%B6%94%EC%84%B8%EC%99%B8%EC%82%BD.ipynb)

# 3-2 · RUL 예측 ① — 추세를 연장해 고장선에 닿는 시간 구하기

**⏱ 80분** · 모듈 3: 열화 지표 → **③ RUL 예측** → 교체 시점 결정

## 🎯 이번 시간의 질문

> 지금까지 HI가 올라온 **추세를 앞으로 연장**하면, 언제 고장 기준선에 닿을까요?

가장 직관적인 RUL 예측 방법입니다. 자동차 연료계를 생각해 보세요. "최근 연비로 계속 달리면 남은 연료로 몇 km 더 갈 수 있다"를 계산하는 것과 같습니다.
이 방법은 세 가지 요소로 이루어집니다.

```
① 건전성 지표 HI (3-1에서 만든 전류 이동평균)
② 고장 기준선 (3-1: 학습용 필터의 고장 직전 HI 중앙값 1.07 A)
③ 추세 모형 (최근 N개 점에 직선을 맞춤)
→ 직선이 기준선과 만나는 시각 − 지금 시각 = 예측 RUL
```

**이번 시간에 익힐 패턴:** 최근 N개 고르기(`tail`) · 직선 맞추기(`np.polyfit`) · 예측 vs 실제 그래프 · 구간별 오차 요약(`groupby`)

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "data/filter_windows.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
runs = pd.read_csv('data/filter_runs.csv')
windows = pd.read_csv('data/filter_windows.csv')
main = windows[windows['status'] == 'main'].copy()
main['hi'] = main.groupby('run_id')['current_a'].transform(lambda s: s.rolling(3, min_periods=1).mean())
train = main[main['split'] == 'train']
test = main[main['split'] == 'test']

TH = 1.07                                  # 고장 기준선 (3-1에서 구한 학습용 고장 직전 HI 중앙값)
one = main[main['run_id'] == 24].copy()    # 3-1의 24번 필터 (F = 425초)
F = one['failure_s'].iloc[0]
print('준비 완료 · 기준선', TH, 'A · 24번 필터 F =', F, '초')

## 1. 한 시점에서 RUL 예측하기 (t = 300초)

24번 필터가 지금 **300초**째 돌고 있다고 합시다. 우리는 300초까지의 데이터만 알고, 고장이 언제일지는 모릅니다.

**1단계 · 최근 N개 고르기** — 300초 이하의 행 중 **마지막 10개**(최근 100초)를 `tail(10)`으로 가져옵니다.

**2단계 · 직선 맞추기** — `np.polyfit(x, y, 1)`은 점들에 가장 잘 맞는 직선 `y = a·x + b`를 찾아 기울기 `a`와 절편 `b`를 돌려줍니다. (마지막 `1`은 1차식, 즉 직선이라는 뜻)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
t_now = 300
past = one[one['time_s'] <= t_now].tail(10)
a, b = np.polyfit(past['time_s'], past['hi'], 1)
print('기울기 a:', round(a, 5), 'A/초')
print('현재 HI:', round(past['hi'].iloc[-1], 3), 'A')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**3단계 · 기준선에 닿는 시각** — 직선 `a·t + b = TH`를 t에 대해 풀면 `t = (TH − b) / a` 입니다. 여기서 지금 시각을 빼면 예측 RUL입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
t_cross = (TH - b) / a
pred_rul = t_cross - t_now
print('기준선 도달 예상 시각:', round(t_cross), '초')
print('예측 RUL:', round(pred_rul), '초 / 실제 RUL:', F - t_now, '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 기울기 약 0.00221 A/초, 예측 RUL 약 85초, 실제 125초.

💬 **결과 해설** — HI가 1초에 0.0022 A씩(100초에 0.22 A) 오르고 있으므로, 지금 0.87 A에서 기준선 1.07 A까지 약 85초 남았다고 예측했습니다.
실제로는 125초를 더 버텼으니 **40초 일찍** 예측했습니다. 일찍 예측하는 쪽(과소 예측)은 조금 일찍 교체하는 손해가 있지만, 늦게 예측하는 쪽(과대 예측)보다 안전합니다.

그림으로 확인합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
plt.plot(one['time_s'], one['hi'], color='gray', label='HI (전체 기록)')
plt.plot(past['time_s'], past['hi'], 'o', color='C0', label='예측에 쓴 최근 10개')
future_t = np.linspace(past['time_s'].iloc[0], t_cross, 50)
plt.plot(future_t, a * future_t + b, '--', color='C0', label='연장한 추세선')
plt.axhline(TH, color='red', label=f'고장 기준선 {TH} A')
plt.axvline(F, color='black', linestyle=':', label=f'실제 고장 {F}초')
plt.xlabel('경과 시간 (초)')
plt.ylabel('HI (A)')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 파란 점선(추세)이 빨간 기준선과 만나는 곳이 예측 고장 시각(약 385초)이고, 점선(검정)이 실제 고장(425초)입니다. **그 사이의 간격이 예측 오차**입니다.

## 2. 시간이 흐르면서 예측은 어떻게 변하나

현장에서는 10초마다 새 데이터가 들어오고, 그때마다 RUL을 **다시 예측**합니다. 1절의 계산을 함수로 묶어 두었습니다.
추세가 없거나 내려가면(기울기 ≤ 0) "기준선에 닿지 않는다"는 뜻이므로 예측 상한인 1,000초로 둡니다.

In [ ]:
# ▶ 그대로 실행하세요
def trend_rul(run, t, N=10, TH=1.07, cap=1000):
    """시각 t까지의 최근 N개 HI로 직선을 맞추고, 기준선 TH에 닿을 때까지의 시간(예측 RUL)을 돌려준다."""
    past = run[run['time_s'] <= t].tail(N)
    if len(past) < N:                       # 데이터가 N개보다 적으면 예측하지 않음
        return np.nan
    a, b = np.polyfit(past['time_s'], past['hi'], 1)
    if past['hi'].iloc[-1] >= TH:           # 이미 기준선을 넘었으면 RUL = 0
        return 0.0
    if a <= 0:                              # 오르지 않으면 언제 닿을지 모름 → 상한
        return float(cap)
    return float(min((TH - b) / a - t, cap))

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for t in [150, 200, 250, 300, 350, 400]:
    print(t, '초: 예측', round(trend_rul(one, t)), '초 / 실제', F - t, '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 150초에는 아직 HI가 오르지 않아(기울기 ≤ 0) 1,000초(모름)로 나오고, 200초에는 350초(실제 225초)로 과대 예측, **300초 이후에는 실제와 가까워집니다**(350초: 예측 80 / 실제 75).
고장에 가까워질수록, 열화가 뚜렷해질수록 추세가 분명해져 예측이 정확해집니다. **먼 미래는 원래 불확실합니다.**

모든 시점의 예측을 실제 RUL과 겹쳐 그려 봅니다. 꾸준히 오른 **24번**과, 오른 뒤 오래 평평했던 **18번** 필터를 비교합니다.

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, rid in zip(axes, [24, 18]):
    run = main[main['run_id'] == rid]
    preds = [trend_rul(run, t) for t in run['time_s']]
    ax.plot(run['time_s'], run['RUL_s'], color='black', label='실제 RUL')
    ax.plot(run['time_s'], preds, 'o-', markersize=3, label='추세 외삽 예측')
    ax.set_title(f'{rid}번 필터 (F = {run["failure_s"].iloc[0]}초)')
    ax.set_xlabel('경과 시간 (초)'); ax.set_ylabel('RUL (초)'); ax.set_ylim(-20, 1050)
    ax.legend()
plt.show()

💬 **결과 해설** — 24번은 250초 이후 예측(파랑)이 실제(검정)를 따라 내려옵니다. 반면 18번은 230초쯤 HI가 벌써 기준선(1.07 A)을 넘어 **예측 RUL = 0**이 되었고, 이후 전류가 기준선 근처에서 오르내리자 예측이 0과 1,000초(모름) 사이를 오갑니다. 실제로는 **792초까지** 버텼습니다.
18번처럼 고장 기준선보다 높은 전류에서 오래 버티는 필터가 있기 때문에(3-1의 고장 직전 HI 분포 참고), **하나의 기준선 + 직선 추세**라는 가정이 깨지면 예측이 크게 틀립니다.

## 3. 평가용 8개 필터 전체로 평가하기

한두 개의 예로는 판단할 수 없습니다. 평가용 8개 필터의 **모든 시점**에서 예측하고, 오차 `|예측 − 실제|`를 **실제 RUL 구간별로** 평균합니다.
구간별로 보는 이유: 정비 결정에 중요한 것은 **고장이 가까울 때(0~200초)**의 정확도이기 때문입니다.

In [ ]:
# ▶ 그대로 실행하세요
def evaluate_trend(df, N=10, TH=1.07):
    """df의 모든 필터·모든 시점에서 추세 외삽 RUL을 예측하고 오차를 표로 만든다."""
    rows = []
    for rid, run in df.groupby('run_id'):
        for t, true_rul in zip(run['time_s'], run['RUL_s']):
            pred = trend_rul(run, t, N, TH)
            if not np.isnan(pred):
                rows.append({'run_id': rid, 'time_s': t, 'true': true_rul, 'pred': pred})
    res = pd.DataFrame(rows)
    res['error'] = (res['pred'] - res['true']).abs()
    res['구간'] = pd.cut(res['true'], [0, 100, 200, 300, 5000], right=False,
                        labels=['0~100초', '100~200초', '200~300초', '300초 이상'])
    return res

res = evaluate_trend(test, N=10, TH=1.07)
print('예측한 시점 수:', len(res))

**P3 · 구간별 평균 오차** — `groupby('구간')`으로 실제 RUL 구간마다 평균 오차를 구합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
res.groupby('구간')['error'].mean().round()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


### 비교 기준: 센서 없이 "보통 수명 − 지금 시간"

새 방법이 쓸 만한지 알려면 **아주 단순한 방법과 비교**해야 합니다. 센서를 전혀 보지 않고 "학습용 필터의 수명 중앙값(425초) − 지금 시간"으로 RUL을 찍는 방법을 **기준 모델(baseline)**로 둡니다(음수는 0으로).

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
life = runs[runs['split'] == 'train']['failure_s'].median()
base = test.copy()
base['pred'] = np.maximum(0, life - base['time_s'])
base['error'] = (base['pred'] - base['RUL_s']).abs()
base['구간'] = pd.cut(base['RUL_s'], [0, 100, 200, 300, 5000], right=False,
                    labels=['0~100초', '100~200초', '200~300초', '300초 이상'])
pd.DataFrame({'추세 외삽': res.groupby('구간')['error'].mean().round(),
              '기준(수명 중앙값 − t)': base.groupby('구간')['error'].mean().round()})
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 추세 외삽 148 / 319 / 268 / 443초, 기준 95 / 126 / 178 / 582초.

💬 **결과 해설** — 솔직한 결과입니다. 이 설정(최근 10개, 기준선 1.07 A)의 추세 외삽은 고장이 가까운 구간에서 **센서를 안 보는 기준보다도 오차가 큽니다.** 먼 구간(300초 이상)에서만 기준보다 낫습니다.
이유는 2절에서 본 두 가지입니다: ① 전류가 평평해지면 추세가 사라지거나 뒤집혀 예측이 크게 흔들리고, ② 필터마다 고장 순간의 HI가 달라 하나의 기준선이 맞지 않습니다.
**설정(N, 기준선)을 바꾸면 나아질까요?** → 과제에서 확인합니다.

---
## 4. 실습 과제 (35분)

⚠️ 원래는 설정을 **학습용** 필터로 골라야 합니다. 여기서는 비교를 체험하기 위해 평가용 결과를 봅니다.

### 📝 과제 1 · 추세를 보는 창 길이 N
N = **5, 10, 20**(최근 50초, 100초, 200초)으로 바꿔 구간별 오차를 비교하세요. 기준선은 1.07 그대로.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for n in [5, 10, 20]: r = evaluate_trend(test, N=n, TH=1.07); print(r.groupby('구간')['error'].mean().round())


### 📝 과제 2 · 고장 기준선 TH
N = 20으로 두고 기준선을 **1.0, 1.07, 1.12** A로 바꿔 보세요. 기준선을 낮추면(보수적으로 잡으면) 고장 직전 오차는 어떻게 되나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: evaluate_trend(test, N=20, TH=1.0)


✏️ **나의 답:** 과제 1·2에서 가장 좋았던 설정은? 기준 모델(95 / 126 / 178 / 582)을 이겼나요? 왜 그 설정이 좋았을까요?

### 📝 과제 3 · 가장 크게 틀린 필터 찾기
N=10, TH=1.07 결과(`res`)에서 **필터별 평균 오차**를 구해 가장 크게 틀린 필터를 찾고, 2절의 그림 코드를 참고해 그 필터의 HI를 그려 원인을 설명하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: res.groupby('run_id')['error'].mean().sort_values()
# 힌트 2: run = main[main['run_id'] == 번호]; plt.plot(run['time_s'], run['hi']); plt.axhline(TH, color='red')


✏️ **나의 답:**

### 📝 과제 4 · 생각해 보기 — 추세 외삽의 가정
추세 외삽 RUL이 잘 맞으려면 무엇이 성립해야 할까요? 이 필터 데이터에서 **깨진 가정**은 무엇인가요? 두 가지 이상 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 추세 외삽 RUL | HI + 고장 기준선 + 최근 추세(직선) → 기준선 도달 시각 − 지금 |
| `np.polyfit(x, y, 1)` | 점들에 가장 잘 맞는 직선의 기울기·절편 |
| 예측의 변화 | 고장이 가까워질수록 정확해진다. 먼 미래는 원래 불확실하다 |
| 기준 모델 비교 | 센서 없는 "수명 중앙값 − t"와 비교해야 방법의 가치를 알 수 있다 |
| 한계 | 열화가 꾸준하지 않거나 고장 기준선이 제각각이면 크게 틀린다. 설정을 사람이 찾아야 한다 |

다음: **3-3 · RUL 예측 ② — 과거 고장 이력에서 머신러닝으로 배우기**